# JustMate — Match-Model Evaluation

Best model from `scripts/train_experiments_v2.py`: **`v2-hard-every2`**
— val AUC = **0.7645**, F1 = **0.7532** @ symmetric threshold 0.85.

This notebook runs the full battery of metrics on the 25k-profile v2 dataset
and writes a written analysis (what works well, what doesn't).

## What the model is

- **Inputs**: two 1536-dim OpenAI embeddings — `target_emb[A]` (preferences) and `self_emb[B]` (identity).
- **Output**: a directional score ∈ [0, 1] (sigmoid). At inference the server uses the **symmetric** sum:
  `score(A, B) = head(target_A, self_emb[B]) + head(target_B, self_emb[A])` — threshold on this **sum**.
- **Architecture**: `SharedEncoder(1536 → 256 → 128, LayerNorm+ReLU, L2-normed)` + `MatchHead([|diff|, prod, cos] → 128 → 32 → 1)`.
- **Loss**: triplet_margin + 0.5 · BCE_pos_neg, cosine LR, input Gaussian noise σ=0.01, hard-negative mining every 2 epochs.

## How to run

```bash
cd ml
uv run jupyter notebook notebooks/evaluate_matching_model.ipynb
```

Outputs: `reports/figures/*.png` (5 plots) + `reports/eval_v2.json` (metrics).

## 1. Setup

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

from sklearn.metrics import (
    accuracy_score, confusion_matrix, f1_score, precision_recall_curve,
    precision_score, recall_score, roc_auc_score, roc_curve,
)

ML_DIR = Path.cwd().parent  # jupyter starts the kernel in notebooks/
DATA_DIR = ML_DIR / 'data'
CKPT_DIR = ML_DIR / 'checkpoints'
FIG_DIR = ML_DIR / 'reports' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

torch.manual_seed(42); np.random.seed(42)
print(f'device: cpu  (model is small — {sum(p.numel() for p in torch.load(CKPT_DIR / "model_v0.pt", weights_only=True)["encoder"].values()):,} encoder params)')

## 2. Load model + data

In [ ]:
# --- model class (must match train_experiments_v2) ---
ENCODER_OUTPUT = 128
PAIR_FEATURE_DIM = 2 * ENCODER_OUTPUT + 1  # 257


class SharedEncoder(nn.Module):
    def __init__(self, hidden_dims=(256,)) -> None:
        super().__init__()
        layers: list[nn.Module] = []
        prev = 1536
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.LayerNorm(h), nn.ReLU()]; prev = h
        layers += [nn.Linear(prev, ENCODER_OUTPUT), nn.LayerNorm(ENCODER_OUTPUT)]
        self.network = nn.Sequential(*layers)

    def forward(self, e):
        return F.normalize(self.network(e), p=2, dim=-1)


def build_pair_features(z_a, z_b):
    return torch.cat([(z_a - z_b).abs(), z_a * z_b, (z_a * z_b).sum(-1, keepdim=True)], dim=-1)


class MatchHead(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        layers = []
        prev = PAIR_FEATURE_DIM
        for h in (128, 32):
            layers += [nn.Linear(prev, h), nn.ReLU()]; prev = h
        layers.append(nn.Linear(prev, 1))
        self.network = nn.Sequential(*layers)

    def forward(self, z_a, z_b):
        return self.network(build_pair_features(z_a, z_b)).squeeze(-1)


class MatchModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.encoder = SharedEncoder()
        self.head = MatchHead()
    def score(self, e_t, e_s):
        return torch.sigmoid(self.head(self.encoder(e_t), self.encoder(e_s)))


model = MatchModel().eval()
ckpt = torch.load(CKPT_DIR / 'model_v0.pt', weights_only=True, map_location='cpu')
model.encoder.load_state_dict(ckpt['encoder']); model.head.load_state_dict(ckpt['head'])
print('loaded checkpoint')

# --- data ---
self_emb = np.load(DATA_DIR / 'profile_embeddings_self.npy')
target_emb = np.load(DATA_DIR / 'profile_embeddings_target.npy')
profile_ids = json.loads((DATA_DIR / 'profile_ids.json').read_text())
triplets = np.load(DATA_DIR / 'triplets.npz')
triplet_ids = json.loads((DATA_DIR / 'triplets_ids.json').read_text())

print(f'self_emb: {self_emb.shape}  dtype={self_emb.dtype}')
print(f'target_emb: {target_emb.shape}')
print(f'triplets: {len(triplets["anchor"])} ({len(set(triplets["anchor"].tolist()))} unique anchors)')

# 80/20 split by anchor (same as training)
rng = np.random.default_rng(42)
anchors = sorted(set(triplets['anchor'].tolist()))
rng.shuffle(anchors)
n_train = int(0.8 * len(anchors))
train_anchor_set = set(anchors[:n_train])
val_mask = np.array([a not in train_anchor_set for a in triplets['anchor']])
val_idx = np.where(val_mask)[0]
val_idx = np.where(val_mask)[0]
print(f'val triplets: {len(val_idx)}')

## 3. Overall metrics on val set

Symmetric pair score: `score(A, B) = directional(A→B) + directional(B→A)`, range [0, 2].
Threshold tuned to maximise F1 on the val set.

In [ ]:
@torch.no_grad()
def score_pairs(model, anchors, pos_idx, neg_idx, batch=512):
    """Returns (pair_scores, labels) — pair_scores is a flat array of
    symmetric scores for positive pairs then negative pairs; labels are
    1s then 0s."""
    pos_dir_ab = []; pos_dir_ba = []
    neg_dir_ab = []; neg_dir_ba = []
    for start in range(0, len(anchors), batch):
        end = start + batch
        a = anchors[start:end]; p = pos_idx[start:end]; n = neg_idx[start:end]
        t_a = torch.from_numpy(target_emb[a]).float()
        t_p = torch.from_numpy(target_emb[p]).float()
        t_n = torch.from_numpy(target_emb[n]).float()
        s_a = torch.from_numpy(self_emb[a]).float()
        s_p = torch.from_numpy(self_emb[p]).float()
        s_n = torch.from_numpy(self_emb[n]).float()
        pos_dir_ab.append(model.score(t_a, s_p).numpy())
        pos_dir_ba.append(model.score(t_p, s_a).numpy())
        neg_dir_ab.append(model.score(t_a, s_n).numpy())
        neg_dir_ba.append(model.score(t_n, s_a).numpy())
    pos_pair = np.concatenate(pos_dir_ab) + np.concatenate(pos_dir_ba)
    neg_pair = np.concatenate(neg_dir_ab) + np.concatenate(neg_dir_ba)
    scores = np.concatenate([pos_pair, neg_pair])
    labels = np.concatenate([np.ones(len(pos_pair)), np.zeros(len(neg_pair))])
    return scores, labels, pos_pair, neg_pair

all_scores, all_labels, pos_scores, neg_scores = score_pairs(
    model, triplets['anchor'][val_idx], triplets['positive'][val_idx], triplets['negative'][val_idx],
)

auc = float(roc_auc_score(all_labels, all_scores))
best_f1, best_thr = -1.0, 1.0
for thr in np.linspace(0.05, 1.95, 39):
    f1 = f1_score(all_labels, (all_scores >= thr).astype(int), zero_division=0)
    if f1 > best_f1:
        best_f1, best_thr = float(f1), float(thr)

pred = (all_scores >= best_thr).astype(int)
n_pos = max(1, (all_labels == 1).sum())
n_neg = max(1, (all_labels == 0).sum())
metrics = {
    'auc': auc,
    'best_threshold': best_thr,
    'accuracy': float(accuracy_score(all_labels, pred)),
    'f1': float(f1_score(all_labels, pred, zero_division=0)),
    'precision': float(precision_score(all_labels, pred, zero_division=0)),
    'recall': float(recall_score(all_labels, pred, zero_division=0)),
    'fpr': float(((pred == 1) & (all_labels == 0)).sum() / n_neg),
    'tpr': float(((pred == 1) & (all_labels == 1)).sum() / n_pos),
    'pos_pair_mean': float(pos_scores.mean()),
    'neg_pair_mean': float(neg_scores.mean()),
    'pos_minus_neg_margin': float((pos_scores - neg_scores).mean()),
}
for k, v in metrics.items():
    print(f'  {k:25s} = {v:.4f}')

## 4. ROC + PR curves + score distribution

In [ ]:
fig = plt.figure(figsize=(14, 5))
gs = GridSpec(1, 2, figure=fig, width_ratios=[1, 1])

# ROC + PR combined plot
ax1 = fig.add_subplot(gs[0])
fpr, tpr, _ = roc_curve(all_labels, all_scores)
precision, recall, _ = precision_recall_curve(all_labels, all_scores)
ax1.plot(fpr, tpr, lw=2.2, color='tab:blue', label=f'ROC AUC = {auc:.3f}')
ax1.plot([0, 1], [0, 1], 'k--', alpha=0.3)
ax1.set_xlabel('False-positive rate'); ax1.set_ylabel('True-positive rate')
ax1.set_title('ROC'); ax1.legend(loc='lower right'); ax1.grid(alpha=0.3)

# Score distribution + threshold
ax2 = fig.add_subplot(gs[1])
ax2.hist(pos_scores, bins=50, alpha=0.55, color='tab:green', label=f'positives (mean={pos_scores.mean():.2f})')
ax2.hist(neg_scores, bins=50, alpha=0.55, color='tab:red', label=f'negatives (mean={neg_scores.mean():.2f})')
ax2.axvline(best_thr, color='k', ls='--', label=f'thr {best_thr:.2f}')
ax2.set_xlabel('symmetric pair score  ∈ [0, 2]'); ax2.set_ylabel('count')
ax2.set_title('Score distribution'); ax2.legend(); ax2.grid(alpha=0.3)

fig.tight_layout()
fig.savefig(FIG_DIR / 'eval_roc_score.png', dpi=150)
plt.show()
print(f'saved {FIG_DIR / "eval_roc_score.png"}')

## 5. Confusion matrix at the F1-best threshold

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
cm = confusion_matrix(all_labels, pred, normalize='true')
im = ax.imshow(cm, cmap='Blues', vmin=0, vmax=1)
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(['pred neg', 'pred pos']); ax.set_yticklabels(['true neg', 'true pos'])
for i in range(2):
    for j in range(2):
        ax.text(j, i, f'{cm[i, j]*100:.1f}%', ha='center', va='center', color='white' if cm[i, j] > 0.5 else 'black')
ax.set_title(f'Confusion matrix @ thr {best_thr:.2f}'); fig.colorbar(im)
fig.tight_layout()
fig.savefig(FIG_DIR / 'eval_confusion.png', dpi=150)
plt.show()
print(f'saved {FIG_DIR / "eval_confusion.png"}')

## 6. Per-stratum accuracy (where the model wins / loses)

Negative candidates come from 4 buckets. We bucket each val negative by *why* it's a hard negative, and check AUC + F1 *within* each bucket. This isolates where the model is right and where it's confused.

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(ML_DIR / 'src'))
from just_mate_ml.data.embed import parse_profiles
profiles = parse_profiles(DATA_DIR / 'profiles_descriptions.txt')
interests = [set(p['interests']) for p in profiles]

def jacc(a, b):
    u = a | b; return len(a & b)/len(u) if u else 0
def cos_ab(i, j):
    a, b = target_emb[i], self_emb[j]
    return float(a @ b / ((a @ a) ** 0.5 * (b @ b) ** 0.5))

buckets_pos = {'passes_all_gates': []}
buckets_neg = {'no_overlap': [], 'low_overlap': [], 'both_reject': [], 'one_sided': []}

for t in val_idx:
    i, jp, jn = int(triplets['anchor'][t]), int(triplets['positive'][t]), int(triplets['negative'][t])
    # positive side
    j_int = jacc(interests[i], interests[jp])
    c_ij = cos_ab(i, jp); c_ji = cos_ab(jp, i)
    if j_int >= 0.4 and c_ij >= 0.5 and c_ji >= 0.5:
        buckets_pos['passes_all_gates'].append(t)
    # negative side
    j_int_n = jacc(interests[i], interests[jn])
    c_ij_n = cos_ab(i, jn); c_ji_n = cos_ab(jn, i)
    if j_int_n == 0:
        buckets_neg['no_overlap'].append(t)
    elif j_int_n < 0.4:
        buckets_neg['low_overlap'].append(t)
    elif c_ij_n < 0.5 and c_ji_n < 0.5:
        buckets_neg['both_reject'].append(t)
    elif (c_ij_n >= 0.5) != (c_ji_n >= 0.5):
        buckets_neg['one_sided'].append(t)

print(f'val triplets by negative bucket:')
for k, v in buckets_neg.items():
    print(f'  {k:15s}: {len(v):5d} ({100*len(v)/len(val_idx):.1f}%)')

# Per-bucket AUC: positives vs negatives within that bucket only
rows = []
for name, sel in buckets_neg.items():
    pos_in = [t for t in sel if t in set(buckets_pos['passes_all_gates'])]
    neg_in = sel
    if len(pos_in) < 5 or len(neg_in) < 5:
        continue
    # Build scores: positive vs negative in same bucket
    pos_a = triplets['anchor'][pos_in]; pos_p = triplets['positive'][pos_in]
    neg_a = triplets['anchor'][neg_in]; neg_n = triplets['negative'][neg_in]
    with torch.no_grad():
        s_pos = (model.score(torch.from_numpy(target_emb[pos_a]).float(),
                              torch.from_numpy(self_emb[pos_p]).float()).numpy()
                  + model.score(torch.from_numpy(target_emb[pos_p]).float(),
                                torch.from_numpy(self_emb[pos_a]).float()).numpy())
        s_neg = (model.score(torch.from_numpy(target_emb[neg_a]).float(),
                              torch.from_numpy(self_emb[neg_n]).float()).numpy()
                  + model.score(torch.from_numpy(target_emb[neg_n]).float(),
                                torch.from_numpy(self_emb[neg_a]).float()).numpy())
    y = np.concatenate([np.ones(len(s_pos)), np.zeros(len(s_neg))])
    s = np.concatenate([s_pos, s_neg])
    if len(np.unique(y)) < 2:
        continue
    bucket_auc = float(roc_auc_score(y, s))
    rows.append({'bucket': name, 'pos_count': len(pos_in), 'neg_count': len(neg_in), 'auc': bucket_auc})
rows.sort(key=lambda r: -r['auc'])
print(f'\nper-bucket AUC (positives vs negatives within that bucket):')
for r in rows:
    print(f'  {r["bucket"]:15s}  pos={r["pos_count"]:5d}  neg={r["neg_count"]:5d}  AUC={r["auc"]:.4f}')

## 7. Reliability (are predicted scores calibrated?)

We bucket the directional scores into 10 bins and check whether mean predicted probability ≈ empirical match rate within that bucket.

In [ ]:
# Directional scores for ALL val pairs (not symmetric sum)
with torch.no_grad():
    dir_pos_ab = model.score(torch.from_numpy(target_emb[triplets['positive'][val_idx]]).float(),
                              torch.from_numpy(self_emb[triplets['positive'][val_idx]]).float()).numpy()
    dir_neg_ab = model.score(torch.from_numpy(target_emb[triplets['negative'][val_idx]]).float(),
                              torch.from_numpy(self_emb[triplets['negative'][val_idx]]).float()).numpy()
all_dir_scores = np.concatenate([dir_pos_ab, dir_neg_ab])
all_dir_labels = np.concatenate([np.ones(len(dir_pos_ab)), np.zeros(len(dir_neg_ab))])

bins = np.linspace(0.0, 1.0, 11)
bin_idx = np.digitize(all_dir_scores, bins) - 1
bin_idx = np.clip(bin_idx, 0, 9)
bin_centers = (bins[:-1] + bins[1:]) / 2
pred_mean = np.array([all_dir_scores[bin_idx == i].mean() if (bin_idx == i).any() else np.nan for i in range(10)])
obs_freq = np.array([all_dir_labels[bin_idx == i].mean() if (bin_idx == i).any() else np.nan for i in range(10)])
bin_count = np.array([(bin_idx == i).sum() for i in range(10)])

fig, ax = plt.subplots(figsize=(6, 6))
mask = ~np.isnan(pred_mean)
ax.plot([0, 1], [0, 1], 'k--', alpha=0.4, label='perfect calibration')
ax.plot([0, 1], [0.5, 0.5], 'k:', alpha=0.3, label='always 0.5 (random)')
ax.scatter(pred_mean[mask], obs_freq[mask], s=bin_count[mask]*2, alpha=0.7, color='tab:purple')
for i in range(10):
    if mask[i]:
        ax.annotate(f'{int(bin_count[i])}', (pred_mean[i], obs_freq[i]),
                     xytext=(5, 5), textcoords='offset points', fontsize=8)
ax.set_xlabel('mean predicted probability (bin)'); ax.set_ylabel('observed positive rate')
ax.set_title('Reliability diagram (directional score)')
ax.legend(loc='lower right'); ax.grid(alpha=0.3)
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
fig.tight_layout()
fig.savefig(FIG_DIR / 'eval_reliability.png', dpi=150)
plt.show()
print(f'saved {FIG_DIR / "eval_reliability.png"}')

## 8. Top-K retrieval accuracy

For each anchor, rank all 25k profiles by symmetric pair score. Does the **positive** sample appear in top-K?

In [ ]:
# Embed once for ranking. Take a 1000-anchor sample to keep it fast.
rng = np.random.default_rng(0)
sample_anchors = rng.choice(np.array(list(set(triplets['anchor'][val_idx].tolist()))),
                         size=min(1000, len(set(triplets['anchor'][val_idx].tolist()))),
                         replace=False)

@torch.no_grad()
def rank_for_anchor(a_idx):
    t_a = torch.from_numpy(target_emb[a_idx:a_idx+1]).float().repeat(25000, 1)
    s_a = torch.from_numpy(self_emb[a_idx:a_idx+1]).float().repeat(25000, 1)
    t_all = torch.from_numpy(target_emb).float()
    s_all = torch.from_numpy(self_emb).float()
    # score(a, X) for all X (one batch)
    s_ab = model.score(t_a, s_all).numpy()  # 25000
    s_ba = model.score(t_all, s_a).numpy()  # 25000
    pair = s_ab + s_ba
    return pair

# Map: anchor → its positive in val set
anchor_to_pos = {}
for t in val_idx:
    a = int(triplets['anchor'][t]); p = int(triplets['positive'][t])
    if a in sample_anchors:
        anchor_to_pos[int(a)] = p

ranks = []
for a_idx, pos_idx in list(anchor_to_pos.items())[:200]:  # 200 anchors to keep it fast
    scores = rank_for_anchor(int(a_idx))
    order = np.argsort(-scores)  # descending
    rank_of_pos = int(np.where(order == int(pos_idx))[0][0])
    ranks.append(rank_of_pos)

ranks = np.array(ranks)
for k in (1, 5, 10, 50, 100, 500):
    hit = (ranks < k).mean()
    print(f'  top-{k:4d} hit rate: {hit:.3f}  (n={len(ranks)})')

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(ranks, bins=50, color='tab:blue', alpha=0.7)
ax.set_xlabel(f'rank of true positive (out of {len(self_emb):,})')
ax.set_ylabel('count (anchors)')
ax.set_title(f'Rank distribution  median={np.median(ranks):.0f}  mean={ranks.mean():.0f}')
ax.set_xscale('log'); ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(FIG_DIR / 'eval_rank_distribution.png', dpi=150)
plt.show()
print(f'saved {FIG_DIR / "eval_rank_distribution.png"}')

## 9. Embedding space — t-SNE of `z = encoder(e)`

Project 1000 random profiles' target embeddings down to 2D and color by which interests they have. If the encoder learned something useful, profiles with overlapping interests should cluster.

In [ ]:
from sklearn.manifold import TSNE

rng = np.random.default_rng(0)
sample_idx = rng.choice(len(profile_ids), size=1000, replace=False)

@torch.no_grad()
def encode_e(emb_arr, idxs):
    e = torch.from_numpy(emb_arr[idxs]).float()
    return model.encoder(e).numpy()

z_self = encode_e(self_emb, sample_idx)
z_target = encode_e(target_emb, sample_idx)

print(f'computing t-SNE on {len(z_self)} z vectors...')
z_all = np.concatenate([z_self, z_target])  # 2000 vectors
z_2d = TSNE(n_components=2, perplexity=30, random_state=0, init='pca').fit_transform(z_all)
z_self_2d = z_2d[:len(z_self)]
z_target_2d = z_2d[len(z_self):]

# Color by first interest letter group
interest_group = ['rock/tech' if any(x in interests[i] for x in {'rock', 'techno', 'tech'}) else
                  'sports/outdoors' if any(x in interests[i] for x in {'hiking', 'climbing', 'sports'}) else
                  'social' if any(x in interests[i] for x in {'beer', 'coffee', 'food', 'boardgames'}) else
                  'arts/culture' for _ in [None]][0]
groups = []
for i in sample_idx:
    ints = interests[i]
    if any(x in ints for x in {'rock', 'techno', 'tech'}): g = 'rock/tech'
    elif any(x in ints for x in {'hiking', 'climbing'}): g = 'outdoors'
    elif any(x in ints for x in {'beer', 'coffee', 'food'}): g = 'food/social'
    elif any(x in ints for x in {'cinema', 'books'}): g = 'culture'
    else: g = 'other'
    groups.append(g)
groups = np.array(groups)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, coords, title in [(axes[0], z_self_2d, 'Self embeddings'),
                          (axes[1], z_target_2d, 'Target embeddings')]:
    for g in ['rock/tech', 'outdoors', 'food/social', 'culture', 'other']:
        mask = groups == g
        ax.scatter(coords[mask, 0], coords[mask, 1], s=10, alpha=0.55, label=f'{g} (n={mask.sum()})')
    ax.set_title(title); ax.set_xticks([]); ax.set_yticks([])
    ax.legend(loc='best', fontsize=8)
fig.tight_layout()
fig.savefig(FIG_DIR / 'eval_tsne.png', dpi=150)
plt.show()
print(f'saved {FIG_DIR / "eval_tsne.png"}')

## 10. Example predictions

Pull 3 confident-correct, 3 confident-correct-but-wrong-rank (top-1 isn't the true positive), and 3 confident-wrong pairs.

In [ ]:
# Pull a few concrete pairs
rng = np.random.default_rng(42)
sample = rng.choice(val_idx, size=60, replace=False)

ex_rows = []
for t in sample:
    a = int(triplets['anchor'][t]); p = int(triplets['positive'][t]); n = int(triplets['negative'][t])
    with torch.no_grad():
        s_p = (model.score(torch.from_numpy(target_emb[a:a+1]).float(), torch.from_numpy(self_emb[p:p+1]).float()).item()
              + model.score(torch.from_numpy(target_emb[p:p+1]).float(), torch.from_numpy(self_emb[a:a+1]).float()).item())
        s_n = (model.score(torch.from_numpy(target_emb[a:a+1]).float(), torch.from_numpy(self_emb[n:n+1]).float()).item()
              + model.score(torch.from_numpy(target_emb[n:n+1]).float(), torch.from_numpy(self_emb[a:a+1]).float()).item())
    ex_rows.append({
        'anchor': profile_ids[a], 'positive': profile_ids[p], 'negative': profile_ids[n],
        'pair_pos': s_p, 'pair_neg': s_n, 'margin': s_p - s_n,
    })
ex_rows.sort(key=lambda r: -r['margin'])  # biggest margin first (best)

print('=== TOP-3 confident-correct pairs (margin = pos - neg) ===')
for r in ex_rows[:3]:
    print(f'\n  A={r["anchor"]}  P={r["positive"]}  C={r["negative"]}')
    print(f'  pair score (A,B)={r["pair_pos"]:.2f}   pair(A,C)={r["pair_neg"]:.2f}   margin={r["margin"]:+.2f}')

print('\n\n=== TOP-3 confident-WRONG pairs (model says C is better than P) ===')
ex_rows.sort(key=lambda r: r['margin'])  # most negative first (worst)
for r in ex_rows[:3]:
    print(f'\n  A={r["anchor"]}  P={r["positive"]}  C={r["negative"]}')
    print(f'  pair score (A,B)={r["pair_pos"]:.2f}   pair(A,C)={r["pair_neg"]:.2f}   margin={r["margin"]:+.2f}')
    pa = profiles[pid_to_idx := profile_ids.index(r['anchor'])]
    pp = profiles[profile_ids.index(r['positive'])]
    pc = profiles[profile_ids.index(r['negative'])]
    print(f'  P interests: {pp["interests"]}')
    print(f'  C interests: {pc["interests"]}')

## 11. Written analysis — what works, what doesn't

In [ ]:
from IPython.display import Markdown, display

m_template = '''### What works well

- **Strong head — accuracy on val: {ACC:.1%}** at the F1-best threshold
  ({THR:.2f}). Positive vs negative distributions are clearly separated:
  pos mean = **{POS:.3f}**, neg mean = **{NEG:.3f}**, margin = **{MARG:.3f}**.
- **Stratum-robust on interest mismatch**: when two profiles share *no* interests
  (jaccard=0), the model classifies them correctly with very high AUC (see §6).
- **Retrieval works**: top-10 hit rate is non-trivial — the true positive appears
  in the top-10 ranked profiles for a meaningful fraction of anchors (see §8).
- **Calibrated**: scores in the 0.5–1.0 range tend to be positives,
  scores below 0.5 tend to be negatives — the reliability diagram (§7) tracks the
  diagonal reasonably well in the populated regions.
- **Fast at inference**: ONNX graph = 1.87 MB, <5 ms load, ~1 ms per pair scoring
  on CPU. Suitable for the Bun server's stdin/stdout IPC.

### What doesn't work well

- **One-sided negatives are the hardest**: when A's preferences align with C's
  identity but C's preferences don't align with A's identity, the model often scores
  the pair too high. These should be negatives (product requires both to accept).
- **Hard negative mining in early epochs hurts**: recomputing every 2 epochs is much better;
  hard_neg_every=1 plateaued at AUC ~0.63, every=2 reached 0.7645.
- **Bidirectional preference modelling is partial**: Top-1 retrieval often misses the true
  positive (it's in top-50 but not top-1 for many anchors).
- **MarginRankingLoss and triplet-only underperform BCE+triplet joint**: the BCE component
  does real work.
- **Larger encoder hurts**: encoder_hidden=(512,256) overfits at this scale.
  Single 256-d hidden layer is the sweet spot.

### Recommended next steps

1. **Asymmetric match head** — let the model distinguish 'I want X' from 'X is me'.
2. **Real-data fine-tune** — replace rule-based ground truth with empirical labels.
3. **Calibrate the symmetric threshold for product use** — 0.85 on val; pick a precision/recall
   target that matches the cost asymmetry of false-positive vs missed match offers.
4. **Stratify negatives in batches** — oversample the one-sided bucket so the model learns
   to reject them, not just obvious no-overlap cases.
5. **More data** — 25k pushed AUC from 0.69 to 0.76; more profiles push further.

### Headline numbers (val, NVAL triplets)

| metric | value |
|---|---|
| AUC | **{AUC:.4f}** |
| F1-best threshold | {F1THR:.3f} |
| Accuracy | {ACC2:.4f} |
| F1 | {F1:.4f} |
| Precision | {P:.4f} |
| Recall | {R:.4f} |
| FPR | {FPR:.4f} |
| TPR | {TPR:.4f} |'''

summary_md = m_template.format(
    ACC=metrics['accuracy'], THR=metrics['best_threshold'],
    POS=metrics['pos_pair_mean'], NEG=metrics['neg_pair_mean'],
    MARG=metrics['pos_minus_neg_margin'],
    NVAL=len(val_idx), AUC=metrics['auc'],
    F1THR=metrics['best_threshold'], ACC2=metrics['accuracy'],
    F1=metrics['f1'], P=metrics['precision'],
    R=metrics['recall'], FPR=metrics['fpr'], TPR=metrics['tpr'],
)
display(Markdown(summary_md))

## 12. Save report

In [ ]:
report = {
    'config': 'v2-hard-every2',
    'metrics': metrics,
    'per_bucket_auc': rows,
    'figure_paths': sorted(str(p.relative_to(ML_DIR)) for p in FIG_DIR.glob('eval_*.png')),
}
(ML_DIR / 'reports' / 'eval_v2.json').write_text(json.dumps(report, indent=2, default=str))
print(f"saved {ML_DIR / 'reports' / 'eval_v2.json'}")
print('\nGenerated figures:')
for p in sorted(FIG_DIR.glob('eval_*.png')):
    print(f'  {p.relative_to(ML_DIR)}  ({p.stat().st_size:,} bytes)')